# Notebook 1 — Ownership Entropy

**Mandate Vacuum Governance Intelligence**

This notebook calculates Ownership Entropy for each complaint category to identify mandate vacuums — areas where no single department clearly owns responsibility.

### Formula
```
H(X) = -Σ pᵢ · log₂(pᵢ)
H_normalized = H(X) / log₂(n)
```
- High entropy (>0.7) → Mandate vacuum
- Low entropy (<0.4) → Clear ownership

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy.stats import entropy
import warnings
warnings.filterwarnings('ignore')

print('Libraries loaded successfully.')

In [ ]:
# Load sample data
df = pd.read_csv('../sample_data/bbmp_complaints_cleaned.csv')n
print(f'Dataset shape: {df.shape}')
print(f'Complaint categories: {df["category"].unique()}')
print(f'Departments involved: {set(df["to_dept"].unique()) | set(df["from_dept"].unique())}')
print('\nFirst 5 rows:')
df.head()

In [ ]:
def ownership_entropy(df, category):
    """
    Calculate Ownership Entropy for a complaint category.
    
    Parameters:
        df       : DataFrame with complaint transfer records
        category : Complaint category string
    
    Returns:
        dict with entropy metrics and department distribution
    """
    cat_df = df[df['category'] == category]
    
    # Count transfers received by each department
    dept_counts = cat_df['to_dept'].value_counts()
    probs = dept_counts / dept_counts.sum()
    
    # Shannon entropy (base 2)
    H = entropy(probs, base=2)
    
    # Normalize by maximum possible entropy
    n_depts = len(dept_counts)
    H_max = np.log2(n_depts) if n_depts > 1 else 1.0
    H_normalized = round(H / H_max, 4) if H_max > 0 else 0.0
    
    # Risk classification
    if H_normalized >= 0.70:
        risk = 'HIGH - Mandate Vacuum'
        color = 'red'
    elif H_normalized >= 0.40:
        risk = 'MEDIUM - Unclear Ownership'
        color = 'orange'
    else:
        risk = 'LOW - Stable Ownership'
        color = 'green'
    
    return {
        'category': category,
        'entropy_raw': round(H, 4),
        'entropy_normalized': H_normalized,
        'num_departments': n_depts,
        'risk_level': risk,
        'color': color,
        'department_distribution': probs.round(3).to_dict()
    }

print('Function defined.')

In [ ]:
# Run entropy analysis for all categories
categories = df['category'].unique()
results = [ownership_entropy(df, cat) for cat in categories]
entropy_df = pd.DataFrame(results)

# Display results table
display_cols = ['category', 'entropy_raw', 'entropy_normalized', 'num_departments', 'risk_level']
print('=== OWNERSHIP ENTROPY RESULTS ===')
print(entropy_df[display_cols].to_string(index=False))

print('\n=== DEPARTMENT DISTRIBUTIONS ===')
for r in results:
    print(f"\n{r['category']}:")
    for dept, prob in r['department_distribution'].items():
        print(f'  {dept}: {prob*100:.1f}%')

In [ ]:
# Visualization — Entropy Bar Chart
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Chart 1: Entropy by category
ax1 = axes[0]
bars = ax1.bar(
    entropy_df['category'],
    entropy_df['entropy_normalized'],
    color=entropy_df['color'],
    edgecolor='white',
    linewidth=1.5,
    alpha=0.85
)
ax1.axhline(y=0.70, color='red', linestyle='--', alpha=0.6, label='High Risk (0.70)')
ax1.axhline(y=0.40, color='orange', linestyle='--', alpha=0.6, label='Medium Risk (0.40)')

# Add value labels on bars
for bar, val in zip(bars, entropy_df['entropy_normalized']):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
             f'{val:.2f}', ha='center', va='bottom', fontsize=10, fontweight='bold')

ax1.set_xlabel('Complaint Category', fontsize=11)
ax1.set_ylabel('Normalized Ownership Entropy', fontsize=11)
ax1.set_title('Mandate Ownership Entropy\nby Complaint Category', fontsize=13, fontweight='bold')
ax1.set_ylim(0, 1.15)
ax1.legend(fontsize=9)

# Chart 2: Department distribution for highest entropy category
ax2 = axes[1]
highest = entropy_df.loc[entropy_df['entropy_normalized'].idxmax()]
dist = highest['department_distribution']
wedge_colors = ['#e74c3c', '#e67e22', '#f39c12', '#27ae60', '#2980b9']
ax2.pie(
    list(dist.values()),
    labels=list(dist.keys()),
    autopct='%1.1f%%',
    colors=wedge_colors[:len(dist)],
    startangle=90
)
ax2.set_title(f'Transfer Distribution\n"{highest["category"]}" (Entropy={highest["entropy_normalized"]})',
              fontsize=13, fontweight='bold')

plt.tight_layout()
plt.savefig('../results/01_ownership_entropy.png', dpi=150, bbox_inches='tight')
plt.show()
print('Chart saved to results/01_ownership_entropy.png')

In [ ]:
# Summary findings
print('=== KEY FINDINGS ===')
for _, row in entropy_df.iterrows():
    emoji = '🔴' if 'HIGH' in row['risk_level'] else '🟠' if 'MEDIUM' in row['risk_level'] else '🟢'
    print(f"{emoji} {row['category']:12s} | Entropy: {row['entropy_normalized']:.2f} | {row['risk_level']}")

high_risk = entropy_df[entropy_df['entropy_normalized'] >= 0.70]
print(f'\n{len(high_risk)} of {len(entropy_df)} categories have CRITICAL mandate vacuums.')
print('These require immediate departmental mandate reassignment.')